# Exactly how much money can marketing afford to spend to acquire a new customer?

## There are two ways to build LTV: 
Predictive (using machine learning to guess the future) 
and 
Historical (calculating the actual value generated based on past behavior). We are building a rock-solid Historical LTV model.

To calculate a customer's true Historical LTV, we need to multiply three distinct ingredients together:

### Average Order Value (AOV): When they swipe their card, how much do they typically spend?

### Purchase Frequency: How many orders do they place in a standard time window (we will use a monthly rate)?

### Lifespan: How many months do they actually remain an active customer before churning?

### (LTV = AOV × Monthly Frequency × Lifespan in Months)

# Step 1: Pull data from DuckDB with connect.py

In [1]:
# Connect to the warehouse
import pandas as pd
from connect import get_warehouse_connection

con = get_warehouse_connection('../mean_mug_analytics/mean_mug.duckdb')

# Get the orders table as DF, display the head of DF.
df = con.execute('SELECT * FROM fct_orders').df()
df.head()

Successfully connected to warehouse: ../mean_mug_analytics/mean_mug.duckdb


,order_id,customer_id,store_location_id,promo_id,order_date,order_time,payment_type_lower,total_amount,total_items_in_basket
0,1,1,1,<NA>,2024-01-26,09:21:00,app,7.50,2
1,2,1,2,2,2024-02-12,07:54:00,app,4.25,1
2,3,2,2,2,2024-02-08,09:22:00,app,16.00,4
3,4,2,1,<NA>,2024-03-26,17:06:00,card,4.25,1
4,5,3,2,<NA>,2024-03-12,13:19:00,cash,10.00,3


In [2]:
# create LTV df
df_ltv = df.groupby('customer_id').agg(
    AOV = ('total_amount',"mean"),
    last_order = ('order_date', 'max'),
    first_order = ('order_date', 'min'),
    total_orders = ('order_id', 'nunique'),
    total_revenue = ('total_amount', 'sum')
    )
# Pandas is treating lifespan_days as an object, not a raw number. If try to divide a timedelta by an integer, Pandas will throw an error, fix is dt.days.
# We assume every customer is alive for at least 1 day. Add 1 to every customer's lifespan before we do any math, avoiding 0 lifespan days.
df_ltv['lifespan_days'] = (df_ltv['last_order'] - df_ltv['first_order']).dt.days
df_ltv.head()

,AOV,last_order,first_order,total_orders,total_revenue,lifespan_days
customer_id,,,,,,
1,5.875,2024-02-12,2024-01-26,2,11.75,17
2,10.125,2024-03-26,2024-02-08,2,20.25,47
3,17.000,2024-03-25,2024-03-12,3,51.00,13
4,4.250,2024-03-24,2024-03-15,2,8.50,9
5,16.250,2024-03-30,2024-03-06,2,32.50,24


## The marketing team doesn't want to know Customer 1's past revenue. They want to know: "On average, when we acquire a completely unknown new customer, how much are they worth?

To answer that, we have to transition from Customer-Level metrics to Store-Level metrics.


In [3]:
# The Month Conversion: Create a new column called lifespan_months. 
# Take lifespan_days, add 1 to it (to fix the zero-day bug), and divide the whole thing by 30.
df_ltv['lifespan_months'] = (df_ltv['lifespan_days'] + 1) / 30
df_ltv.head()


,AOV,last_order,first_order,total_orders,total_revenue,lifespan_days,lifespan_months
customer_id,,,,,,,
1,5.875,2024-02-12,2024-01-26,2,11.75,17,0.600000
2,10.125,2024-03-26,2024-02-08,2,20.25,47,1.600000
3,17.000,2024-03-25,2024-03-12,3,51.00,13,0.466667
4,4.250,2024-03-24,2024-03-15,2,8.50,9,0.333333
5,16.250,2024-03-30,2024-03-06,2,32.50,24,0.833333


In [4]:
# The Frequency Rate: Create a new column called monthly_frequency. This is simply total_orders divided by lifespan_months.
df_ltv['monthly_frequency'] = df_ltv['total_orders'] / df_ltv['lifespan_months']
df_ltv.head()

,AOV,last_order,first_order,total_orders,total_revenue,lifespan_days,lifespan_months,monthly_frequency
customer_id,,,,,,,,
1,5.875,2024-02-12,2024-01-26,2,11.75,17,0.600000,3.333333
2,10.125,2024-03-26,2024-02-08,2,20.25,47,1.600000,1.250000
3,17.000,2024-03-25,2024-03-12,3,51.00,13,0.466667,6.428571
4,4.250,2024-03-24,2024-03-15,2,8.50,9,0.333333,6.000000
5,16.250,2024-03-30,2024-03-06,2,32.50,24,0.833333,2.400000


In [5]:
# The Golden Averages: Now, leave the individual rows behind. You need to calculate the overall .mean() across your entire DataFrame for three specific columns:
#Overall Average AOV
average_aov = df_ltv['AOV'].mean()
average_monthly_freq = df_ltv['monthly_frequency'].mean()
average_lifespan_months = df_ltv['lifespan_months'].mean()

# The Final LTV: Multiply those three store-wide averages together. Save that final number as a variable called store_ltv.
# Remember: (LTV = AOV × Monthly Frequency × Lifespan in Months)
store_ltv = average_aov * average_monthly_freq * average_lifespan_months
store_ltv

np.float64(113.91387562235889)

In [6]:
import pandas_gbq
from google.oauth2 import service_account

# 1. Authenticate downloaded JSON key
credentials = service_account.Credentials.from_service_account_file(
    '../mean_mug_analytics/mean-mug-analytics-7f3a37898649.json'
)

In [7]:
# 2. Define the exact Cloud destination
project_id = 'mean-mug-analytics' 
table_id = f'{project_id}.mean_mug_marts.mart_customer_ltv'

In [8]:
# 3. Blast it to BigQuery
print("Pushing to Google Cloud...")
pandas_gbq.to_gbq(
    df_ltv.reset_index(), 
    destination_table=table_id,
    project_id=project_id,
    credentials=credentials,
    if_exists='replace' 
)
print("Success! Table is live in BigQuery.")

Pushing to Google Cloud...
Success! Table is live in BigQuery.


In [9]:
con.close()